# EV Charging Hourly Demand Prediction

**AI Services MS2 | Machine Learning Milestone 02 | 60 marks | Exactly 20 test cases**

Estimate the hourly number of charging sessions initiated across a public EV charging network using synthetic observations. This notebook uses three already-split CSVs in the same folder: `ev_train.csv`, `ev_test.csv` and `ev_predict.csv`. Calendar and weather inputs must be available for the hour being estimated. The split is chronological, so the test period follows training. Intermediate outputs are generated only by running the final workflow cells.

## 1. Imports and feature definitions
The model uses four original weather measurements, four engineered numerical time features, and four categorical indicators. Exclude `session_count`, `guest_sessions`, `member_sessions`, `observation_id`, and the raw `observation_date` from the predictors.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

TARGET = "session_count"
RAW_NUMERIC = ["temperature_c", "feels_like_c", "humidity_pct", "wind_speed_kmph", "hour"]
NUMERIC_FEATURES = [
    "temperature_c", "feels_like_c", "humidity_pct", "wind_speed_kmph",
    "month", "is_weekend", "hour_sin", "hour_cos"
]
CATEGORICAL = ["season", "weather_condition", "holiday", "working_day"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL


## 2. Common loader and time-based features
**`load_data(filename: str) -> pd.DataFrame`**

Reuse for every input CSV. Keep first `observation_id`, remove invalid dates, normalize categories, and compute month, weekend indicator and cyclical hour features.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Clean hourly records and engineer the same features for all CSVs."""
    df = pd.read_csv(filename)
    df = df.drop_duplicates(subset="observation_id", keep="first")

    df["observation_date"] = pd.to_datetime(
        df["observation_date"], format="mixed", dayfirst=True, errors="coerce"
    )
    df = df.dropna(subset=["observation_date"]).copy()

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df[RAW_NUMERIC] = df[RAW_NUMERIC].replace([np.inf, -np.inf], np.nan)

    for col in CATEGORICAL:
        df[col] = (
            df[col].astype("string").str.strip().str.lower()
            .replace("", pd.NA)
            .str.replace(r"^([+-]?\d+)\.0+$", r"\1", regex=True)
            .fillna("unknown").astype(str)
        )

    df["month"] = df["observation_date"].dt.month
    df["is_weekend"] = (df["observation_date"].dt.dayofweek >= 5).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    return df


## 3. Reusable Scikit-learn pipeline
**`build_pipeline(algorithm: BaseEstimator) -> Pipeline`**

Use training-fitted median imputation, standard scaling, and one-hot encoding with unseen-category tolerance.

In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Build an unfitted pipeline with training-only numeric/category preparation."""
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("category", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", algorithm)
    ])


## 4. Prepare chronological labeled datasets
**`prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]`**

Remove invalid or negative session-count targets. Return `X_train, X_test, y_train, y_test`. Keep the original order; do not random-shuffle or fit transformations on the test set.

In [ ]:
def prepare_datasets(
    train_file: str, test_file: str
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Return clean predictor/target pairs for chronological train/test files."""
    train_df = load_data(train_file)
    test_df = load_data(test_file)
    for df in (train_df, test_df):
        df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
        df[TARGET] = df[TARGET].replace([np.inf, -np.inf], np.nan)
        df.dropna(subset=[TARGET], inplace=True)
        df.drop(df[df[TARGET] < 0].index, inplace=True)

    X_train = train_df[FEATURES]
    X_test = test_df[FEATURES]
    y_train = train_df[TARGET]
    y_test = test_df[TARGET]
    return X_train, X_test, y_train, y_test


## 5. Train candidate regressors
**`train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]`**

Fit two independent full pipelines: Decision Tree and Random Forest with the required parameters.

In [ ]:
def train_models(
    X_train: pd.DataFrame, y_train: pd.Series
) -> dict[str, Pipeline]:
    """Fit two separate complete demand-estimation pipelines."""
    models = {
        "decision_tree": DecisionTreeRegressor(
            max_depth=10, min_samples_leaf=4, random_state=42
        ),
        "random_forest": RandomForestRegressor(
            n_estimators=120, max_depth=14, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )
    }
    trained_models = {}
    for name, algorithm in models.items():
        model_pipeline = build_pipeline(algorithm)
        model_pipeline.fit(X_train, y_train)
        trained_models[name] = model_pipeline
    return trained_models


## 6. Evaluate held-back predictions
**`evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame`**

Calculate MAE, RMSE and RMSLE using nonnegative predictions, and rank from smallest RMSLE to largest.

In [ ]:
def evaluate_models(
    trained_models: dict[str, Pipeline],
    X_test: pd.DataFrame, y_test: pd.Series
) -> pd.DataFrame:
    """Score fitted pipelines and rank by lowest RMSLE."""
    results = []
    for name, model_pipeline in trained_models.items():
        predictions = np.maximum(model_pipeline.predict(X_test), 0)
        results.append({
            "model_name": name,
            "mae": mean_absolute_error(y_test, predictions),
            "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
            "rmsle": np.sqrt(mean_squared_log_error(y_test, predictions))
        })
    comparison = pd.DataFrame(results)
    return comparison.sort_values("rmsle").reset_index(drop=True)


## 7. Select the best model
**`select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]`**

Return the first-ranked model and its existing fitted pipeline, without refitting.

In [ ]:
def select_best_model(
    comparison: pd.DataFrame, trained_models: dict[str, Pipeline]
) -> tuple[str, Pipeline]:
    """Select the first model from the RMSLE-ranked comparison."""
    best_name = comparison.iloc[0]["model_name"]
    best_model = trained_models[best_name]
    return best_name, best_model


## 8. Predict new hourly sessions
**`predict_demand(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame`**

Accept cleaned queued input, select only the required features, clip negatives to zero, round to whole sessions and retain surviving record IDs.

In [ ]:
def predict_demand(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Return nonnegative whole-session demand predictions for queued rows."""
    predictions = np.maximum(model.predict(df[FEATURES]), 0)
    predictions = np.rint(predictions).astype(int)
    return pd.DataFrame({
        "observation_id": df["observation_id"].to_numpy(),
        "predicted_sessions": predictions
    })


## 9. Training workflow
**`train() -> tuple[str, pd.DataFrame]`**

Only this function fits estimators. It saves `best_model.pkl` and `model_comparison.csv` after a training run.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Train, compare and save the best fitted pipeline and report."""
    X_train, X_test, y_train, y_test = prepare_datasets(
        "ev_train.csv", "ev_test.csv"
    )
    trained_models = train_models(X_train, y_train)
    comparison = evaluate_models(trained_models, X_test, y_test)
    best_name, best_model = select_best_model(comparison, trained_models)
    joblib.dump(best_model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)
    print(comparison.round(4).to_string(index=False))
    print("Best model:", best_name)
    return best_name, comparison


## 10. Independent inference workflow
**`predict() -> pd.DataFrame`**

Reloads `best_model.pkl` and writes `ev_predictions.csv` without fitting.

In [ ]:
def predict() -> pd.DataFrame:
    """Load the saved pipeline and estimate queued sessions without retraining."""
    queued = load_data("ev_predict.csv")
    best_model = joblib.load("best_model.pkl")
    result = predict_demand(best_model, queued)
    result.to_csv("ev_predictions.csv", index=False)
    print(result.head().to_string(index=False))
    return result


## Run training
Requires `ev_train.csv` and `ev_test.csv`. Run once initially, or when refreshing the model.

In [ ]:
best_name, comparison = train()

## Run prediction independently
Requires `ev_predict.csv` and `best_model.pkl`. In a new Jupyter session, execute definition cells, then this cell only.

In [ ]:
ev_predictions = predict()